In [44]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine

df = pd.read_parquet("vate_validate_student_only.parquet")

# Extract embedding column names
fraud_cols = [col for col in df.columns if col.startswith('fraud_student_') and col != 'fraud_student_']
real_cols = [col for col in df.columns if col.startswith('real_student_') and col != 'real_student_']

# Sort by numeric suffix to ensure correct order
fraud_cols = sorted(fraud_cols, key=lambda x: int(x.split('_')[2]))
real_cols = sorted(real_cols, key=lambda x: int(x.split('_')[2]))

# print out the average cosine similarity between the two embeddings for all rows with label 0
label_0_df = df[df['label'] == 0]
cosine_similarities_0 = []
for index, row in label_0_df.iterrows():
    fraud_emb = row[fraud_cols].to_numpy()
    real_emb = row[real_cols].to_numpy()
    # compute cosine similarity
    cos_sim = 1 - cosine(fraud_emb, real_emb)
    cosine_similarities_0.append(cos_sim)
print(f"Average Cosine Similarity (label 0): {np.mean(cosine_similarities_0)}")

# then, print out the average cosine similarity between the two embeddings for all rows with label 1
label_1_df = df[df['label'] == 1]
cosine_similarities_1 = []
for index, row in label_1_df.iterrows():
    fraud_emb = row[fraud_cols].to_numpy()
    real_emb = row[real_cols].to_numpy()
    # compute cosine similarity
    cos_sim = 1 - cosine(fraud_emb, real_emb)
    cosine_similarities_1.append(cos_sim)
print(f"Average Cosine Similarity (label 1): {np.mean(cosine_similarities_1)}")


Average Cosine Similarity (label 0): 0.8650052623191464
Average Cosine Similarity (label 1): 0.953007966295877


In [46]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine

df = pd.read_parquet("vate_validate.parquet")

# Extract embedding column names
fraud_cols = [col for col in df.columns if col.startswith('fraud_emb_') and col != 'fraud_emb_']
real_cols = [col for col in df.columns if col.startswith('real_emb_') and col != 'real_emb_']

# Sort by numeric suffix to ensure correct order
fraud_cols = sorted(fraud_cols, key=lambda x: int(x.split('_')[2]))
real_cols = sorted(real_cols, key=lambda x: int(x.split('_')[2]))

# print out the average cosine similarity between the two embeddings for all rows with label 0
label_0_df = df[df['label'] == 0]
cosine_similarities_0 = []
for index, row in label_0_df.iterrows():
    fraud_emb = row[fraud_cols].to_numpy()
    real_emb = row[real_cols].to_numpy()
    # compute cosine similarity
    cos_sim = 1 - cosine(fraud_emb, real_emb)
    cosine_similarities_0.append(cos_sim)
print(f"Average Cosine Similarity (label 0): {np.mean(cosine_similarities_0)}")

# then, print out the average cosine similarity between the two embeddings for all rows with label 1
label_1_df = df[df['label'] == 1]
cosine_similarities_1 = []
for index, row in label_1_df.iterrows():
    fraud_emb = row[fraud_cols].to_numpy()
    real_emb = row[real_cols].to_numpy()
    # compute cosine similarity
    cos_sim = 1 - cosine(fraud_emb, real_emb)
    cosine_similarities_1.append(cos_sim)
print(f"Average Cosine Similarity (label 1): {np.mean(cosine_similarities_1)}")


Average Cosine Similarity (label 0): 0.896699178030674
Average Cosine Similarity (label 1): 0.963398800966964


In [47]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine

df = pd.read_parquet("golden_embeddings_validate.parquet")

# Extract embedding column names
fraud_cols = [col for col in df.columns if col.startswith("fraud_aligned_") and col != "fraud_aligned_"]
real_cols  = [col for col in df.columns if col.startswith("real_aligned_") and col != "real_aligned_"]

# Sort by numeric suffix
fraud_cols = sorted(fraud_cols, key=lambda x: int(x.split("_")[2]))
real_cols  = sorted(real_cols, key=lambda x: int(x.split("_")[2]))

def safe_cosine_similarity(a, b, eps=1e-12):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na < eps or nb < eps:
        return np.nan   # skip bad rows
    return 1 - cosine(a, b)

# label 0
label_0_df = df[df["label"] == 0]
cosine_similarities_0 = []
for _, row in label_0_df.iterrows():
    fraud_emb = row[fraud_cols].to_numpy(dtype=np.float32)
    real_emb  = row[real_cols].to_numpy(dtype=np.float32)

    cos_sim = safe_cosine_similarity(fraud_emb, real_emb)
    cosine_similarities_0.append(cos_sim)

print(f"Average Cosine Similarity (label 0): {np.nanmean(cosine_similarities_0)}")
print(f"Skipped rows (label 0): {np.isnan(cosine_similarities_0).sum()}")

# label 1
label_1_df = df[df["label"] == 1]
cosine_similarities_1 = []
for _, row in label_1_df.iterrows():
    fraud_emb = row[fraud_cols].to_numpy(dtype=np.float32)
    real_emb  = row[real_cols].to_numpy(dtype=np.float32)

    cos_sim = safe_cosine_similarity(fraud_emb, real_emb)
    cosine_similarities_1.append(cos_sim)

print(f"Average Cosine Similarity (label 1): {np.nanmean(cosine_similarities_1)}")
print(f"Skipped rows (label 1): {np.isnan(cosine_similarities_1).sum()}")


Average Cosine Similarity (label 0): 0.8065789341926575
Skipped rows (label 0): 0
Average Cosine Similarity (label 1): 0.952099084854126
Skipped rows (label 1): 0


In [49]:
import re
import numpy as np
import pandas as pd

df = pd.read_parquet("validate_pairs_with_siglip_embeddings.parquet")

def _sorted_prefixed_cols(df: pd.DataFrame, prefix: str):
    cols = [c for c in df.columns if isinstance(c, str) and c.startswith(prefix)]
    if not cols:
        raise KeyError(f"No columns found with prefix '{prefix}'")
    def key_fn(c: str):
        suf = c[len(prefix):]
        return int(suf) if re.fullmatch(r"-?\d+", suf) else 10**18
    return sorted(cols, key=lambda c: (key_fn(c), c))

def l2_normalize(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(n, eps)

def pair_cos_and_dist(fraud: np.ndarray, real: np.ndarray):
    f = l2_normalize(fraud.astype(np.float32, copy=False))
    r = l2_normalize(real.astype(np.float32, copy=False))
    cos = np.sum(f * r, axis=1)
    cos = np.clip(cos, -1.0, 1.0)
    d = np.sqrt(np.maximum(2.0 - 2.0 * cos, 0.0))
    return cos, d

def pick_mpos_mneg(df, fraud_prefix, real_prefix, label_col="label",
                   q_pos=0.75, q_neg=0.25, min_gap=0.02):
    y = df[label_col].astype(int).to_numpy()

    fcols = _sorted_prefixed_cols(df, fraud_prefix)
    rcols = _sorted_prefixed_cols(df, real_prefix)
    if len(fcols) != len(rcols):
        raise ValueError(f"Dim mismatch: {len(fcols)} fraud cols vs {len(rcols)} real cols")

    F = df[fcols].to_numpy(dtype=np.float32, copy=False)
    R = df[rcols].to_numpy(dtype=np.float32, copy=False)

    _, d = pair_cos_and_dist(F, R)
    d_pos = d[y == 1]
    d_neg = d[y == 0]

    m_pos = float(np.quantile(d_pos, q_pos))
    m_neg = float(np.quantile(d_neg, q_neg))

    # ensure a real band
    if m_neg < m_pos + min_gap:
        m_neg = m_pos + min_gap

    print("---- chosen margins ----")
    print(f"m_pos = q{int(q_pos*100)}(d_pos) = {m_pos:.6f}")
    print(f"m_neg = q{int(q_neg*100)}(d_neg) = {m_neg:.6f}")
    print("---- active fractions ----")
    print(f"pos_active_frac = mean(d_pos > m_pos): {float(np.mean(d_pos > m_pos)):.3f}")
    print(f"neg_active_frac = mean(d_neg < m_neg): {float(np.mean(d_neg < m_neg)):.3f}")

    return m_pos, m_neg

# Example:
m_pos, m_neg = pick_mpos_mneg(df, fraud_prefix="fraud_emb_", real_prefix="real_emb_", label_col="label",
                              q_pos=0.75, q_neg=0.25)

import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoSidedMarginLoss(nn.Module):
    def __init__(self, m_pos: float, m_neg: float):
        super().__init__()
        self.m_pos = float(m_pos)
        self.m_neg = float(m_neg)

    def forward(self, z_text, z_teacher, y):
        # assume you normalize outside or do it here:
        z_text = F.normalize(z_text, dim=1)
        z_teacher = F.normalize(z_teacher, dim=1)

        y = y.float()
        d = torch.norm(z_text - z_teacher, dim=1)  # in [0,2]

        pos = y * torch.relu(d - self.m_pos).pow(2)        # only if positives too far
        neg = (1.0 - y) * torch.relu(self.m_neg - d).pow(2) # only if negatives too close

        return (pos + neg).mean()


---- chosen margins ----
m_pos = q75(d_pos) = 0.494830
m_neg = q25(d_neg) = 0.634133
---- active fractions ----
pos_active_frac = mean(d_pos > m_pos): 0.250
neg_active_frac = mean(d_neg < m_neg): 0.250
